In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
def compute_cumulative_outcome_fractions(df, masses, N):
    """
    df      : DataFrame with columns ['mass', 'time', 'particle_id', 'result']
              where result is 'Ejected' or 'Captured', and each row represents
              the time at which that particle's outcome occurred
    masses  : list/array of unique mass values, in the same order as N
    N       : list/array of total particle counts, one per mass in `masses`

    Returns
    -------
    ejection_df, capture_df : DataFrames indexed by mass, columns = sorted
                               unique time values, entries = CUMULATIVE
                               fraction of N with that result by that time
                               (i.e. fraction ejected/captured at or before
                               each time, monotonically non-decreasing
                               across each row)
    """
    if len(masses) != len(N):
        raise ValueError("masses and N must be the same length")

    mass_to_N = dict(zip(masses, N))

    times = np.sort(df["time"].unique())

    ejection_df = pd.DataFrame(index=masses, columns=times, dtype=float)
    capture_df = pd.DataFrame(index=masses, columns=times, dtype=float)

    for mass in masses:
        mass_df = df[df["mass"] == mass]
        denom = mass_to_N[mass]

        counts = (
            mass_df.groupby(["time", "result"])
            .size()
            .unstack(fill_value=0)
        )

        # reindex to the full sorted time axis, filling any missing times
        # with 0 new outcomes at that time, then cumsum across time
        ejected_counts = counts.get("Ejected", pd.Series(0, index=counts.index))
        captured_counts = counts.get("Captured", pd.Series(0, index=counts.index))

        ejected_counts = ejected_counts.reindex(times, fill_value=0)
        captured_counts = captured_counts.reindex(times, fill_value=0)

        ejection_df.loc[mass] = (ejected_counts.cumsum() / denom).values
        capture_df.loc[mass] = (captured_counts.cumsum() / denom).values

    return ejection_df, capture_df

In [3]:
2215154

2215154

In [13]:
df_f_1=pd.read_csv('/Users/haydenmonk/Downloads/HPCC/Mass_Fraction/Outputs/Ejection_Results/2215154_ejection_results.txt', header=None)
df_f_2=pd.read_csv('/Users/haydenmonk/Downloads/HPCC/Mass_Fraction/Outputs/Ejection_Results_Restarted/2215154/2215154_mass_7.94328235e-05_restarted_results.txt', header=None)

In [14]:
df_f_2.columns=['mass','time','result','particle_id']
df_f_1.columns=['mass','time','result','particle_id']

In [16]:
df_f=pd.concat([df_f_1,df_f_2])

In [23]:
df_f['mass'].unique()

array([1.58489319e-04, 7.94328235e-05, 3.98107171e-05, 1.99526231e-05,
       1.00000000e-05])

In [22]:
target = 7.94328235e-05

df_f.loc[np.isclose(df_f['mass'], target), 'mass'] = target

In [24]:
df_e,df_c=compute_cumulative_outcome_fractions(df_f, df_f['mass'].unique(),[800,800,800,800,800])

In [25]:
df_e.sort_index(ascending=False)

,0.000000e+00,3.099754e+04,3.799849e+04,7.899374e+04,7.999366e+04,1.219976e+05,1.719864e+05,3.429932e+05,4.239664e+05,4.339828e+05,...,1.230251e+07,1.233451e+07,1.234851e+07,1.350046e+07,1.359746e+07,1.370546e+07,1.392045e+07,1.394045e+07,1.417344e+07,1.468342e+07
0.000158,0.0,0.00125,0.00125,0.00125,0.00125,0.00125,0.00250,0.00250,0.00375,0.00375,...,0.27875,0.27875,0.27875,0.27875,0.27875,0.27875,0.27875,0.27875,0.27875,0.27875
0.000079,0.0,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00125,...,0.25750,0.25875,0.26000,0.26125,0.26250,0.26375,0.26500,0.26625,0.26750,0.26750
0.000040,0.0,0.00000,0.00000,0.00000,0.00000,0.00125,0.00125,0.00125,0.00125,0.00125,...,0.04500,0.04500,0.04500,0.04500,0.04500,0.04500,0.04500,0.04500,0.04500,0.04500
0.000020,0.0,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,...,0.01250,0.01250,0.01250,0.01250,0.01250,0.01250,0.01250,0.01250,0.01250,0.01250
0.000010,0.0,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,...,0.00625,0.00625,0.00625,0.00625,0.00625,0.00625,0.00625,0.00625,0.00625,0.00625
